# Simuladores: El modelo de Rescorla y Wagner### Capítulo 11 — *Aprendizaje y Comportamiento Adaptable: Principios y Modelos***Arturo Bouzas** · Facultad de Psicología, UNAM · bouzaslab25.com---Ejecutar las celdas en orden. Cada simulador es interactivo: los controles actualizan la gráfica en tiempo real.Estos simuladores acompañan al Capítulo 11, donde derivamos cómo el error de predicción compartido —la diferencia entre el SBI obtenido y la suma de todos los valores presentes— genera competencia entre estímulos como consecuencia matemática de una sola ecuación.

---## Simulador 11.1 — Explorador de EnsombrecimientoCuando dos estímulos A y B se presentan simultáneamente seguidos de un SBI, el modelo de Rescorla y Wagner predice que la **proporción de saliencias** ($\beta_A / \beta_B$) determina cómo se distribuye el valor predictivo entre ellos. En este simulador explorarás esa relación.**Antes de manipular los controles, predice:**- Si $\beta_A = 0.6$ y $\beta_B = 0.3$, ¿qué fracción del valor total capturará A?- ¿Qué ocurre si ambos estímulos tienen la misma saliencia?- ¿Cambiar α afecta la *distribución* final o solo la *velocidad* con que se alcanza?Verifica tus predicciones con el simulador.

In [ ]:
#@title **Simulador 11.1** — Explorador de Ensombrecimiento
# ============================================================
# Simulador 11.1 — Explorador de Ensombrecimiento
# Capítulo 11: Modelo de Rescorla y Wagner
# Aprendizaje y Comportamiento Adaptable: Principios y Modelos
# ============================================================

# IMPORTACIONES
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import warnings
warnings.filterwarnings("ignore")

# ── Paletas claro / oscuro ────────────────────────────────────
_PALETAS_111 = {
    'claro': dict(
        azul       = '#2C5282',
        naranja    = '#C05621',
        verde      = '#276749',
        gris       = '#718096',
        fig_bg     = 'white',
        ax_bg      = 'white',
        texto      = '#2D3748',
        legend_bg  = 'white',
        panel_bg   = '#EBF4FF',
        panel_bord = '#2C5282',
        header_bg  = '#2C5282',
        header_fg  = 'white',
        sec_color  = '#2C5282',
    ),
    'oscuro': dict(
        azul       = '#90CDF4',
        naranja    = '#FBD38D',
        verde      = '#9AE6B4',
        gris       = '#A0AEC0',
        fig_bg     = '#1A202C',
        ax_bg      = '#2D3748',
        texto      = '#E2E8F0',
        legend_bg  = '#2D3748',
        panel_bg   = '#1A202C',
        panel_bord = '#4A5568',
        header_bg  = '#2C5282',
        header_fg  = 'white',
        sec_color  = '#90CDF4',
    ),
}

def _p_111(oscuro: bool) -> dict:
    return _PALETAS_111['oscuro' if oscuro else 'claro']


# ── Lógica del modelo ─────────────────────────────────────────
def simular_111(alpha, beta_A, beta_B, n_ensayos):
    V_A, V_B = 0.0, 0.0
    hist_A, hist_B, hist_err = [V_A], [V_B], [1.0]

    for _ in range(n_ensayos):
        V_total = V_A + V_B
        error = 1.0 - V_total
        V_A += alpha * beta_A * error
        V_B += alpha * beta_B * error
        hist_A.append(V_A)
        hist_B.append(V_B)
        hist_err.append(1.0 - (V_A + V_B))

    return np.array(hist_A), np.array(hist_B), np.array(hist_err)


# ── Visualización ────────────────────────────────────────────
def graficar_111(alpha, beta_A, beta_B, n_ensayos, mostrar_tabla, mostrar_error, tema):

    oscuro = (tema == 'Oscuro')
    p = _p_111(oscuro)

    plt.rcParams.update({
        'font.family'       : 'serif',
        'figure.facecolor'  : p['fig_bg'],
        'axes.facecolor'    : p['ax_bg'],
        'axes.edgecolor'    : p['gris'],
        'axes.spines.top'   : False,
        'axes.spines.right' : False,
        'axes.grid'         : True,
        'grid.alpha'        : 0.30,
        'grid.color'        : p['gris'],
        'axes.labelcolor'   : p['gris'],
        'xtick.color'       : p['gris'],
        'ytick.color'       : p['gris'],
        'text.color'        : p['texto'],
        'legend.facecolor'  : p['legend_bg'],
        'legend.edgecolor'  : p['gris'],
        'legend.labelcolor' : p['texto'],
        'axes.labelsize'    : 11,
        'xtick.labelsize'   : 10,
        'ytick.labelsize'   : 10,
    })

    h_A, h_B, h_err = simular_111(alpha, beta_A, beta_B, n_ensayos)
    ensayos = np.arange(len(h_A))

    # Layout: 2 or 3 panels depending on error option
    if mostrar_error:
        fig, (ax1, ax_err, ax2) = plt.subplots(
            1, 3, figsize=(16, 5.5),
            gridspec_kw={'width_ratios': [2, 1.2, 1], 'wspace': 0.30}
        )
    else:
        fig, (ax1, ax2) = plt.subplots(
            1, 2, figsize=(14, 6),
            gridspec_kw={'width_ratios': [2, 1], 'wspace': 0.25}
        )

    fig.patch.set_facecolor(p['fig_bg'])
    for ax in fig.axes:
        ax.set_facecolor(p['ax_bg'])

    mk = 'o' if n_ensayos <= 25 else None

    # Panel izquierdo: Curvas de aprendizaje
    ax1.plot(ensayos, h_A, color=p['azul'], linewidth=2,
             marker=mk, markersize=4, markerfacecolor=p['azul'],
             label=f'$V_A$ ($\\beta_A$ = {beta_A:.2f})')
    ax1.plot(ensayos, h_B, color=p['naranja'], linewidth=2,
             marker=mk, markersize=4, markerfacecolor=p['naranja'],
             label=f'$V_B$ ($\\beta_B$ = {beta_B:.2f})')
    ax1.plot(ensayos, h_A + h_B, color=p['gris'], linewidth=1.5,
             linestyle='--', label='$V_{total}$')

    ax1.axhline(1.0, color=p['verde'], linewidth=1.2, linestyle=':', alpha=0.8)
    ax1.set_ylim(-0.05, 1.15)
    ax1.set_xlabel('Ensayo')
    ax1.set_ylabel('Valor predictivo $V$')
    ax1.set_title('Condicionamiento del Compuesto A+B', color=p['texto'], fontsize=11)
    ax1.legend(fontsize=9, loc='lower right', framealpha=0.9)

    # Panel central (opcional): Error de predicción
    if mostrar_error:
        ax_err.plot(ensayos, h_err, color=p['verde'], linewidth=2,
                    marker=mk, markersize=3, markerfacecolor=p['verde'])
        ax_err.axhline(0, color=p['gris'], linewidth=0.8, alpha=0.5)
        ax_err.set_xlabel('Ensayo')
        ax_err.set_ylabel('$\\delta = (\\lambda - V_{total})$')
        ax_err.set_title('Error de predicción', color=p['texto'], fontsize=11)
        ax_err.set_ylim(-0.1, 1.1)

    # Panel derecho: Barras de distribución asintótica
    v_a_final, v_b_final = h_A[-1], h_B[-1]
    barras = ax2.bar(['$V_A$', '$V_B$'], [v_a_final, v_b_final],
                     color=[p['azul'], p['naranja']], width=0.5, edgecolor=p['ax_bg'])

    ax2.set_ylim(0, 1.15)
    ax2.set_ylabel('Valor final')
    ax2.set_title('Distribución del Crédito', color=p['texto'], fontsize=11)

    ratio = beta_A / (beta_A + beta_B) if (beta_A + beta_B) > 0 else 0
    ax2.axhline(y=ratio, color=p['verde'], ls=':', lw=1.5,
                label=f'Razón teórica = {ratio:.2f}')
    ax2.legend(fontsize=9, loc='upper right')

    for bar, val in zip(barras, [v_a_final, v_b_final]):
        ax2.text(bar.get_x() + bar.get_width()/2, val + 0.03,
                 f'{val:.2f}', ha='center', color=p['texto'], fontsize=10)

    # Título principal
    titulo = (f'α = {alpha:.2f}   |   β_A = {beta_A:.2f}   |   β_B = {beta_B:.2f}   |   '
              f'V_A final = {v_a_final:.3f}   |   V_B final = {v_b_final:.3f}')
    fig.suptitle(titulo, fontsize=10, color=p['gris'], y=1.02, fontweight="bold")

    plt.show()

    # Ecuaciones en Markdown
    md_eq = (
        rf"**Ecuaciones del modelo de Rescorla y Wagner (Adquisición conjunta con $\lambda=1$):**" "\n\n"
        rf"$$V_{{total}} = V_A + V_B \qquad \delta = (\lambda - V_{{total}})$$ " "\n\n"
        rf"$$\Delta V_A = {alpha:.2f} \cdot {beta_A:.2f} \cdot \delta \qquad \qquad \Delta V_B = {alpha:.2f} \cdot {beta_B:.2f} \cdot \delta$$"
    )
    display(Markdown(md_eq))

    if mostrar_tabla and n_ensayos <= 30:
        _mostrar_tabla_numerica_111(alpha, beta_A, beta_B, n_ensayos)


def _mostrar_tabla_numerica_111(alpha, beta_A, beta_B, n_ensayos):
    V_A, V_B = 0.0, 0.0
    filas = [
        "| Ensayo | $V_A$ | $V_B$ | $V_{total}$ | $\\delta = (\\lambda - V_{total})$ | $\\Delta V_A$ | $\\Delta V_B$ |",
        "|:------:|:-----:|:-----:|:-----------:|:---------------------------:|:-------------:|:-------------:|",
    ]

    for t in range(n_ensayos):
        V_total = V_A + V_B
        delta = 1.0 - V_total
        delta_V_A = alpha * beta_A * delta
        delta_V_B = alpha * beta_B * delta

        filas.append(
            f"| {t+1} | {V_A:.4f} | {V_B:.4f} | {V_total:.4f} | {delta:+.4f} | {delta_V_A:+.4f} | {delta_V_B:+.4f} |"
        )
        V_A += delta_V_A
        V_B += delta_V_B

    display(Markdown(
        "\n**Tabla numérica — evolución ensayo a ensayo:**\n\n"
        + "\n".join(filas)
    ))


def _html_header_111(oscuro: bool) -> str:
    p = _p_111(oscuro)
    return (
        f'<div style="'
        f'background-color:{p["header_bg"]};'
        f'color:{p["header_fg"]};'
        f'font-family:Georgia,serif;'
        f'font-size:14px;font-weight:bold;'
        f'padding:8px 14px;'
        f'border-radius:6px 6px 0 0;'
        f'letter-spacing:0.5px;">'
        f'&nbsp;Simulador 11.1 &mdash; Explorador de Ensombrecimiento'
        f'</div>'
    )

def _html_sec_111(texto: str, oscuro: bool) -> str:
    p = _p_111(oscuro)
    return (
        f'<div style="'
        f'color:{p["sec_color"]};'
        f'font-family:Georgia,serif;'
        f'font-size:11px;font-weight:bold;'
        f'text-transform:uppercase;letter-spacing:1px;'
        f'margin:8px 0 2px 4px;">{texto}</div>'
    )

# ── Controles ─────────────────────────────────────────────────
estilo_111   = {'description_width': '180px'}
layout_l_111 = widgets.Layout(width='500px')

w_tema_111 = widgets.ToggleButtons(
    options=['Claro', 'Oscuro'],
    value='Claro',
    description='',
    style={'button_width': '120px'},
    layout=widgets.Layout(width='auto'),
)

w_alpha_111 = widgets.FloatSlider(
    value=0.20, min=0.00, max=1, step=0.05,
    description='α (SBI):', style=estilo_111, layout=layout_l_111,
    readout_format='.2f')

w_beta_A_111 = widgets.FloatSlider(
    value=0.40, min=0.0, max=1, step=0.05,
    description='β_A (Saliencia estímulo A):', style=estilo_111, layout=layout_l_111,
    readout_format='.2f')

w_beta_B_111 = widgets.FloatSlider(
    value=0.20, min=0.0, max=1, step=0.05,
    description='β_B (Saliencia estímulo B):', style=estilo_111, layout=layout_l_111,
    readout_format='.2f')

w_n_111 = widgets.IntSlider(
    value=20, min=2, max=50, step=1,
    description='Número de ensayos:', style=estilo_111, layout=layout_l_111)

w_tabla_111 = widgets.Checkbox(
    value=True, description='Mostrar tabla numérica (ensayos ≤ 30)',
    style=estilo_111, layout=layout_l_111)

w_error_111 = widgets.Checkbox(
    value=False, description='Mostrar curva de error de predicción',
    style=estilo_111, layout=layout_l_111)

btn_ejemplo_111 = widgets.Button(
    description='▶  Cargar ejemplo: Ensombrecimiento clásico',
    button_style='',
    layout=widgets.Layout(width='340px', height='32px'),
)

def _cargar_ejemplo_111(_):
    w_alpha_111.value  = 0.20
    w_beta_A_111.value = 0.40
    w_beta_B_111.value = 0.20
    w_n_111.value      = 30
    w_tabla_111.value  = True
    w_error_111.value  = True

btn_ejemplo_111.on_click(_cargar_ejemplo_111)

w_header_111 = widgets.HTML(value=_html_header_111(False))
w_sec1_111   = widgets.HTML(value=_html_sec_111('Tema', False))
w_sec2_111   = widgets.HTML(value=_html_sec_111('Parámetros de Aprendizaje', False))

_body_layout_111 = widgets.Layout(
    padding='10px 16px 14px 16px',
    border=f'1px solid {_PALETAS_111["claro"]["panel_bord"]}',
    border_radius='0 0 6px 6px',
)

_body_111 = widgets.VBox(
    [
        w_sec1_111, w_tema_111,
        w_sec2_111,
        w_alpha_111,
        w_beta_A_111,
        w_beta_B_111,
        w_n_111,
        widgets.HTML("<br>"),
        w_tabla_111,
        w_error_111,
        widgets.HBox([btn_ejemplo_111]),
    ],
    layout=_body_layout_111,
)

ui_111 = widgets.VBox([w_header_111, _body_111])

def _actualizar_tema_111(change):
    oscuro = (change['new'] == 'Oscuro')
    p      = _p_111(oscuro)

    w_header_111.value = _html_header_111(oscuro)
    w_sec1_111.value   = _html_sec_111('Tema', oscuro)
    w_sec2_111.value   = _html_sec_111('Parámetros de Aprendizaje', oscuro)

    _body_111.layout.border = f'1px solid {p["panel_bord"]}'

w_tema_111.observe(_actualizar_tema_111, names='value')

out_111 = widgets.interactive_output(
    graficar_111,
    {
        'alpha'        : w_alpha_111,
        'beta_A'       : w_beta_A_111,
        'beta_B'       : w_beta_B_111,
        'n_ensayos'    : w_n_111,
        'mostrar_tabla': w_tabla_111,
        'mostrar_error': w_error_111,
        'tema'         : w_tema_111,
    }
)

display(ui_111, out_111)


### Ejercicios — Simulador 11.1**Básico.** Configura $\beta_A = \beta_B = 0.40$ y observa el resultado. ¿Hay ensombrecimiento? ¿Qué fracción del valor total captura cada estímulo? Ahora cambia a $\beta_A = 0.80$, $\beta_B = 0.20$. Antes de mirar la gráfica, predice cuánto capturará cada uno. Verifica tu predicción y formula una regla general que relacione $\beta_A / \beta_B$ con $V_A / V_B$ en el equilibrio.**Intermedio.** Con $\beta_A = 0.40$, $\beta_B = 0.20$, compara los resultados con $\alpha = 0.05$, $\alpha = 0.20$ y $\alpha = 0.50$. ¿Cambia la distribución asintótica del crédito o solo la velocidad con que se alcanza? Activa la curva de error de predicción para observar cómo disminuye $\delta$. ¿Qué relación tiene la velocidad de caída del error con el producto $\alpha \cdot (\beta_A + \beta_B)$?**Avanzado.** Configura $\beta_A = 0.50$, $\beta_B = 0.00$. ¿Qué le ocurre a $V_B$? Ahora configura $\alpha = 0.00$. ¿Qué le ocurre a ambos? Estos casos extremos revelan el papel de cada parámetro: $\beta = 0$ significa que el estímulo es "invisible" para el sistema; $\alpha = 0$ significa que el SBI no impulsa aprendizaje. ¿Qué interpretación biológica darías a cada situación?

---## Simulador 11.2 — Explorador de Protocolos Rescorla-WagnerEste simulador permite explorar los fenómenos centrales del capítulo como **consecuencias de una misma ecuación** aplicada a distintos protocolos experimentales. Ensombrecimiento, bloqueo, sobreexpectación, desbloqueo e inhibición condicionada no son fenómenos independientes — todos emergen del error de predicción compartido.**Antes de explorar cada protocolo, predice:**- *Bloqueo:* Si A ya predice perfectamente el SBI, ¿qué error queda para B?- *Sobreexpectación:* Si $V_A \approx 1$ y $V_B \approx 1$, ¿qué ocurre con $V_{total}$ cuando se presenta el compuesto con un solo SBI?- *Desbloqueo:* ¿Qué ocurriría si en la Fase 2 el SBI fuera *más intenso* que en la Fase 1?- *Inhibición:* ¿Puede un estímulo adquirir valor *negativo*? ¿Bajo qué condiciones?

In [ ]:
#@title **Simulador 11.2** — Explorador de Protocolos Rescorla-Wagner
# ============================================================
# Simulador 11.2 — Explorador de Protocolos Rescorla-Wagner
# Capítulo 11: Modelo de Rescorla y Wagner
# Aprendizaje y Comportamiento Adaptable: Principios y Modelos
# ============================================================

# IMPORTACIONES
import numpy as np
import matplotlib.pyplot as plt
import ipywidgets as widgets
from IPython.display import display, Markdown, clear_output
import warnings
warnings.filterwarnings("ignore")

# ── Paletas claro / oscuro ────────────────────────────────────
_PALETAS_112 = {
    'claro': dict(
        azul       = '#2C5282',
        naranja    = '#C05621',
        verde      = '#276749',
        gris       = '#718096',
        fig_bg     = 'white',
        ax_bg      = 'white',
        texto      = '#2D3748',
        legend_bg  = 'white',
        panel_bg   = '#EBF4FF',
        panel_bord = '#2C5282',
        header_bg  = '#2C5282',
        header_fg  = 'white',
        sec_color  = '#2C5282',
    ),
    'oscuro': dict(
        azul       = '#90CDF4',
        naranja    = '#FBD38D',
        verde      = '#9AE6B4',
        gris       = '#A0AEC0',
        fig_bg     = '#1A202C',
        ax_bg      = '#2D3748',
        texto      = '#E2E8F0',
        legend_bg  = '#2D3748',
        panel_bg   = '#1A202C',
        panel_bord = '#4A5568',
        header_bg  = '#2C5282',
        header_fg  = 'white',
        sec_color  = '#90CDF4',
    ),
}

def _p_112(oscuro: bool) -> dict:
    return _PALETAS_112['oscuro' if oscuro else 'claro']


# ── Descripciones ─────────────────────────────────────────────
_DESCRIPCIONES_112 = {
    'Adquisición simple': 'A → SBI. Un estímulo solo, sin competencia.',
    'Ensombrecimiento': 'A+B → SBI. Dos estímulos compiten desde el inicio.',
    'Bloqueo': 'Fase 1: A → SBI | Fase 2: A+B → SBI. A preentrenado bloquea a B.',
    'Sobreexpectación': 'F1: A→SBI | F2: B→SBI | F3: A+B→SBI. Valores bajan en F3.',
    'Desbloqueo': 'F1: A→SBI (λ=1) | F2: A+B→SBI (λ variable). Cambia λ₂ para desbloquear.',
    'Inhibición condicionada': 'Ensayos intercalados A+ y AB−. B adquiere valor negativo.',
}


# ── Lógica del modelo ─────────────────────────────────────────
def simular_112(protocol_name, alpha, beta_A, beta_B, n_fase, R2):
    betas = {'A': beta_A, 'B': beta_B}
    V = {'A': 0.0, 'B': 0.0}
    registros = []
    marcas_fase = []
    ensayo_global = 1

    def run_phase(phase_name, stims, R_val, n_trials):
        nonlocal ensayo_global
        for _ in range(n_trials):
            v_tot = sum(V[s] for s in stims)
            error = R_val - v_tot

            dV_A = alpha * betas['A'] * error if 'A' in stims else 0.0
            dV_B = alpha * betas['B'] * error if 'B' in stims else 0.0

            registros.append({
                'ensayo': ensayo_global,
                'fase': phase_name,
                'stims': "+".join(stims),
                'R': R_val,
                'V_A_prev': V['A'],
                'V_B_prev': V['B'],
                'V_tot': v_tot,
                'error': error,
                'dV_A': dV_A,
                'dV_B': dV_B,
                'V_A_next': V['A'] + dV_A,
                'V_B_next': V['B'] + dV_B
            })

            V['A'] += dV_A
            V['B'] += dV_B
            ensayo_global += 1

    if protocol_name == 'Adquisición simple':
        run_phase('Fase 1', ['A'], 1.0, n_fase)
    elif protocol_name == 'Ensombrecimiento':
        run_phase('Fase 1', ['A', 'B'], 1.0, n_fase)
    elif protocol_name == 'Bloqueo':
        run_phase('Fase 1', ['A'], 1.0, n_fase)
        marcas_fase.append(len(registros))
        run_phase('Fase 2', ['A', 'B'], R2, n_fase)
    elif protocol_name == 'Sobreexpectación':
        run_phase('Fase 1', ['A'], 1.0, n_fase)
        marcas_fase.append(len(registros))
        run_phase('Fase 2', ['B'], 1.0, n_fase)
        marcas_fase.append(len(registros))
        run_phase('Fase 3', ['A', 'B'], R2, n_fase)
    elif protocol_name == 'Desbloqueo':
        run_phase('Fase 1', ['A'], 1.0, n_fase)
        marcas_fase.append(len(registros))
        run_phase('Fase 2', ['A', 'B'], R2, n_fase)
    elif protocol_name == 'Inhibición condicionada':
        # Ensayos intercalados: A+ y AB-
        for i in range(n_fase):
            # Ensayo A+ (reforzado)
            v_tot = V['A']
            error = 1.0 - v_tot
            dV_A = alpha * betas['A'] * error
            registros.append({
                'ensayo': ensayo_global, 'fase': 'A+',
                'stims': 'A', 'R': 1.0,
                'V_A_prev': V['A'], 'V_B_prev': V['B'],
                'V_tot': v_tot, 'error': error,
                'dV_A': dV_A, 'dV_B': 0.0,
                'V_A_next': V['A'] + dV_A, 'V_B_next': V['B']
            })
            V['A'] += dV_A
            ensayo_global += 1

            # Ensayo AB- (no reforzado)
            v_tot = V['A'] + V['B']
            error = 0.0 - v_tot
            dV_A = alpha * betas['A'] * error
            dV_B = alpha * betas['B'] * error
            registros.append({
                'ensayo': ensayo_global, 'fase': 'AB−',
                'stims': 'A+B', 'R': 0.0,
                'V_A_prev': V['A'], 'V_B_prev': V['B'],
                'V_tot': v_tot, 'error': error,
                'dV_A': dV_A, 'dV_B': dV_B,
                'V_A_next': V['A'] + dV_A, 'V_B_next': V['B'] + dV_B
            })
            V['A'] += dV_A
            V['B'] += dV_B
            ensayo_global += 1

    return registros, marcas_fase


# ── Visualización ────────────────────────────────────────────
def graficar_112(protocolo, alpha, beta_A, beta_B, n_fase, R2, mostrar_tabla, tema):

    oscuro = (tema == 'Oscuro')
    p = _p_112(oscuro)

    plt.rcParams.update({
        'font.family'       : 'serif',
        'figure.facecolor'  : p['fig_bg'],
        'axes.facecolor'    : p['ax_bg'],
        'axes.edgecolor'    : p['gris'],
        'axes.spines.top'   : False,
        'axes.spines.right' : False,
        'axes.grid'         : True,
        'grid.alpha'        : 0.30,
        'grid.color'        : p['gris'],
        'axes.labelcolor'   : p['gris'],
        'xtick.color'       : p['gris'],
        'ytick.color'       : p['gris'],
        'text.color'        : p['texto'],
        'legend.facecolor'  : p['legend_bg'],
        'legend.edgecolor'  : p['gris'],
        'legend.labelcolor' : p['texto'],
        'axes.labelsize'    : 11,
        'xtick.labelsize'   : 10,
        'ytick.labelsize'   : 10,
    })

    registros, marcas = simular_112(protocolo, alpha, beta_A, beta_B, n_fase, R2)

    # Reconstruir trayectorias incluyendo el estado inicial 0.0
    hA = [r['V_A_prev'] for r in registros] + [registros[-1]['V_A_next']]
    hB = [r['V_B_prev'] for r in registros] + [registros[-1]['V_B_next']]

    ensayos = np.arange(len(hA))

    fig, ax = plt.subplots(figsize=(13, 6))
    fig.patch.set_facecolor(p['fig_bg'])
    ax.set_facecolor(p['ax_bg'])

    mk = 'o' if len(ensayos) <= 35 else None
    msize = 4

    ax.plot(ensayos, hA, color=p['azul'], linewidth=2.2,
            marker=mk, markersize=msize, markerfacecolor=p['azul'],
            label=f'$V_A$ ($\\beta_A$={beta_A:.2f})')

    if protocolo != 'Adquisición simple':
        ax.plot(ensayos, hB, color=p['naranja'], linewidth=2.2,
                marker=mk, markersize=msize, markerfacecolor=p['naranja'],
                label=f'$V_B$ ($\\beta_B$={beta_B:.2f})')

        vtot_plot = [a + b for a, b in zip(hA, hB)]
        ax.plot(ensayos, vtot_plot, color=p['gris'], linewidth=1.5,
                linestyle='--', label='$V_{total}$')

    for m in marcas:
        ax.axvline(x=m, color=p['verde'], linestyle=':', linewidth=1.5, alpha=0.8)

    ax.axhline(0, color=p['gris'], linewidth=0.8, alpha=0.5)

    ax.set_xlabel('Ensayos')
    ax.set_ylabel('Valor predictivo $V$')

    ymin = min(min(hA), min(hB)) - 0.1 if protocolo != 'Adquisición simple' else -0.1
    ax.set_ylim(ymin, max(1.2, max(hA) + 0.15))

    ax.legend(fontsize=9, loc='best', framealpha=0.9)

    # Textos de descripción
    desc = _DESCRIPCIONES_112[protocolo]
    ax.set_title(f'{protocolo}\n{desc}', fontsize=11, color=p['texto'])

    # Título principal
    titulo = (f'α = {alpha:.2f}   |   β_A = {beta_A:.2f}   |   β_B = {beta_B:.2f}   |   '
              f'V_A final = {hA[-1]:.3f}   |   V_B final = {hB[-1]:.3f}')
    fig.suptitle(titulo, fontsize=10, color=p['gris'], y=1.02, fontweight="bold")

    plt.tight_layout()
    plt.show()

    md_eq = (
        rf"**Ecuación base — Rescorla y Wagner:**" "\n\n"
        rf"$$\Delta V_i = \alpha \cdot \beta_i \cdot (\lambda - \Sigma V)$$"
    )
    display(Markdown(md_eq))

    total_ensayos = len(registros)
    if mostrar_tabla and total_ensayos <= 30:
        _mostrar_tabla_numerica_112(registros)


def _mostrar_tabla_numerica_112(registros):
    filas = [
        "| Ensayo | Fase | Estímulos | $\\lambda$ | $V_A$ | $V_B$ | $V_{total}$ | $\\delta$ | $\\Delta V_A$ | $\\Delta V_B$ |",
        "|:------:|:----:|:---------:|:---------:|:-----:|:-----:|:-----------:|:--------:|:-------------:|:-------------:|",
    ]

    for r in registros:
        filas.append(
            f"| {r['ensayo']} | {r['fase']} | {r['stims']} | {r['R']:.1f} | {r['V_A_prev']:.3f} | {r['V_B_prev']:.3f} | {r['V_tot']:.3f} | {r['error']:+.3f} | {r['dV_A']:+.3f} | {r['dV_B']:+.3f} |"
        )

    display(Markdown(
        "\n**Tabla numérica — evolución ensayo a ensayo:**\n\n"
        + "\n".join(filas)
    ))


def _html_header_112(oscuro: bool) -> str:
    p = _p_112(oscuro)
    return (
        f'<div style="'
        f'background-color:{p["header_bg"]};'
        f'color:{p["header_fg"]};'
        f'font-family:Georgia,serif;'
        f'font-size:14px;font-weight:bold;'
        f'padding:8px 14px;'
        f'border-radius:6px 6px 0 0;'
        f'letter-spacing:0.5px;">'
        f'&nbsp;Simulador 11.2 &mdash; Explorador de Protocolos Rescorla-Wagner'
        f'</div>'
    )

def _html_sec_112(texto: str, oscuro: bool) -> str:
    p = _p_112(oscuro)
    return (
        f'<div style="'
        f'color:{p["sec_color"]};'
        f'font-family:Georgia,serif;'
        f'font-size:11px;font-weight:bold;'
        f'text-transform:uppercase;letter-spacing:1px;'
        f'margin:8px 0 2px 4px;">{texto}</div>'
    )

# ── Controles ─────────────────────────────────────────────────
estilo_112   = {'description_width': '190px'}
layout_l_112 = widgets.Layout(width='520px')

w_tema_112 = widgets.ToggleButtons(
    options=['Claro', 'Oscuro'],
    value='Claro',
    description='',
    style={'button_width': '120px'},
    layout=widgets.Layout(width='auto'),
)

w_proto_112 = widgets.Dropdown(
    options=['Adquisición simple', 'Ensombrecimiento', 'Bloqueo',
             'Sobreexpectación', 'Desbloqueo', 'Inhibición condicionada'],
    value='Bloqueo',
    description='Protocolo Experimental:', style=estilo_112, layout=layout_l_112)

w_alpha_112 = widgets.FloatSlider(
    value=0.20, min=0.0, max=1, step=0.05,
    description='α:', style=estilo_112, layout=layout_l_112,
    readout_format='.2f')

w_beta_A_112 = widgets.FloatSlider(
    value=0.40, min=0.0, max=1, step=0.05,
    description='β_A (Saliencia estímulo A):', style=estilo_112, layout=layout_l_112,
    readout_format='.2f')

w_beta_B_112 = widgets.FloatSlider(
    value=0.40, min=0.0, max=1, step=0.05,
    description='β_B (Saliencia estímulo B):', style=estilo_112, layout=layout_l_112,
    readout_format='.2f')

w_n_112 = widgets.IntSlider(
    value=10, min=2, max=50, step=1,
    description='Número de ensayos por fase:', style=estilo_112, layout=layout_l_112)

w_R2_112 = widgets.FloatSlider(
    value=1.0, min=0.0, max=2.0, step=0.1,
    description='λ₂ (Refuerzo en Fase 2/3):', style=estilo_112, layout=layout_l_112,
    readout_format='.1f')

w_tabla_112 = widgets.Checkbox(
    value=True, description='Mostrar tabla numérica (Total ensayos ≤ 30)',
    style=estilo_112, layout=layout_l_112)

# ── Botones de ejemplo por protocolo ──────────────────────────
btn_layout = widgets.Layout(width='260px', height='30px')

btn_bloqueo = widgets.Button(description='▶ Bloqueo clásico', layout=btn_layout)
btn_desbloqueo = widgets.Button(description='▶ Desbloqueo (λ₂=1.5)', layout=btn_layout)
btn_sobreexp = widgets.Button(description='▶ Sobreexpectación', layout=btn_layout)
btn_inhibicion = widgets.Button(description='▶ Inhibición condicionada', layout=btn_layout)

def _ej_bloqueo(_):
    w_proto_112.value = 'Bloqueo'; w_alpha_112.value = 0.20
    w_beta_A_112.value = 0.50; w_beta_B_112.value = 0.50
    w_n_112.value = 15; w_R2_112.value = 1.0; w_tabla_112.value = True

def _ej_desbloqueo(_):
    w_proto_112.value = 'Desbloqueo'; w_alpha_112.value = 0.20
    w_beta_A_112.value = 0.50; w_beta_B_112.value = 0.50
    w_n_112.value = 15; w_R2_112.value = 1.5; w_tabla_112.value = True

def _ej_sobreexp(_):
    w_proto_112.value = 'Sobreexpectación'; w_alpha_112.value = 0.20
    w_beta_A_112.value = 0.50; w_beta_B_112.value = 0.50
    w_n_112.value = 15; w_R2_112.value = 1.0; w_tabla_112.value = True

def _ej_inhibicion(_):
    w_proto_112.value = 'Inhibición condicionada'; w_alpha_112.value = 0.20
    w_beta_A_112.value = 0.50; w_beta_B_112.value = 0.50
    w_n_112.value = 15; w_R2_112.value = 1.0; w_tabla_112.value = True

btn_bloqueo.on_click(_ej_bloqueo)
btn_desbloqueo.on_click(_ej_desbloqueo)
btn_sobreexp.on_click(_ej_sobreexp)
btn_inhibicion.on_click(_ej_inhibicion)

# ── Deshabilitar λ₂ cuando no aplica ─────────────────────────
def _toggle_R2(change):
    proto = change['new']
    usa_R2 = proto in ('Bloqueo', 'Sobreexpectación', 'Desbloqueo')
    w_R2_112.disabled = not usa_R2
    if not usa_R2:
        w_R2_112.value = 1.0

w_proto_112.observe(_toggle_R2, names='value')

w_header_112 = widgets.HTML(value=_html_header_112(False))
w_sec1_112   = widgets.HTML(value=_html_sec_112('Tema', False))
w_sec2_112   = widgets.HTML(value=_html_sec_112('Configuración del Protocolo', False))
w_sec3_112   = widgets.HTML(value=_html_sec_112('Parámetros de Aprendizaje', False))
w_sec4_112   = widgets.HTML(value=_html_sec_112('Ejemplos predefinidos', False))

_body_layout_112 = widgets.Layout(
    padding='10px 16px 14px 16px',
    border=f'1px solid {_PALETAS_112["claro"]["panel_bord"]}',
    border_radius='0 0 6px 6px',
)

_body_112 = widgets.VBox(
    [
        w_sec1_112, w_tema_112,
        w_sec2_112,
        w_proto_112,
        w_n_112,
        w_R2_112,
        w_sec3_112,
        w_alpha_112,
        w_beta_A_112,
        w_beta_B_112,
        widgets.HTML("<br>"),
        w_tabla_112,
        w_sec4_112,
        widgets.HBox([btn_bloqueo, btn_desbloqueo]),
        widgets.HBox([btn_sobreexp, btn_inhibicion]),
    ],
    layout=_body_layout_112,
)

ui_112 = widgets.VBox([w_header_112, _body_112])

def _actualizar_tema_112(change):
    oscuro = (change['new'] == 'Oscuro')
    p      = _p_112(oscuro)

    w_header_112.value = _html_header_112(oscuro)
    w_sec1_112.value   = _html_sec_112('Tema', oscuro)
    w_sec2_112.value   = _html_sec_112('Configuración del Protocolo', oscuro)
    w_sec3_112.value   = _html_sec_112('Parámetros de Aprendizaje', oscuro)
    w_sec4_112.value   = _html_sec_112('Ejemplos predefinidos', oscuro)

    _body_112.layout.border = f'1px solid {p["panel_bord"]}'

w_tema_112.observe(_actualizar_tema_112, names='value')

out_112 = widgets.interactive_output(
    graficar_112,
    {
        'protocolo'    : w_proto_112,
        'alpha'        : w_alpha_112,
        'beta_A'       : w_beta_A_112,
        'beta_B'       : w_beta_B_112,
        'n_fase'       : w_n_112,
        'R2'           : w_R2_112,
        'mostrar_tabla': w_tabla_112,
        'tema'         : w_tema_112,
    }
)

display(ui_112, out_112)


### Ejercicios — Simulador 11.2**Básico.** Carga el ejemplo "Bloqueo clásico" y observa la tabla numérica. Identifica el ensayo exacto donde comienza la Fase 2. ¿Cuál es el error de predicción ($\delta$) en ese primer ensayo? ¿Por qué $\Delta V_B$ es esencialmente cero? Escribe en tus propias palabras la explicación del bloqueo en términos del error compartido.**Intermedio.** Carga el ejemplo "Desbloqueo" y observa que $V_B$ ahora sí adquiere valor. Cambia $\lambda_2$ gradualmente de 1.0 a 2.0. ¿Cuánto necesita aumentar $\lambda_2$ para que $V_B$ supere 0.10? ¿Y para que supere 0.25? El desbloqueo fue una de las primeras confirmaciones experimentales del modelo — explica por qué un cambio en la intensidad del SBI "desbloquea" el aprendizaje sobre B.**Avanzado (Inhibición).** Carga el ejemplo "Inhibición condicionada" y observa cómo $V_B$ desciende a valores negativos. Examina la tabla ensayo a ensayo: nota cómo en los ensayos A+ el error es positivo (solo A presente) y en los ensayos AB− el error es negativo (A+B presente, $\lambda=0$). ¿Por qué B *necesita* la presencia de A para volverse inhibidor? Ahora configura $\beta_A = 0.10$ y $\beta_B = 0.50$. ¿Qué ocurre con $V_B$? ¿Cambia la asíntota o solo la velocidad?**Reflexión.** Compara los resultados de Ensombrecimiento y Bloqueo. En ambos, B termina con menos valor del que tendría si se presentara solo. Pero la causa es diferente. En el ensombrecimiento, la competencia ocurre *desde el inicio*; en el bloqueo, A ya agotó el error *antes* de que B apareciera. ¿Es esta una diferencia de grado o de naturaleza? Argumenta tu respuesta usando la ecuación.

---## Créditos y licenciaEste notebook es parte del proyecto:> **Bouzas, A. (2026).** *Aprendizaje y Comportamiento Adaptable: Principios y Modelos.*> Lab25, Facultad de Psicología, UNAM.> https://www.bouzaslab25.comApoyo en la construcción del simulador: **Eduardo Sánchez**.Código disponible en: **https://github.com/bouzaslab25/libro-aca**Licencia: [CC BY-NC-SA 4.0](https://creativecommons.org/licenses/by-nc-sa/4.0/)